# Day 10 · 像管理原始碼一樣管理 Context：壓縮與 Token 最佳化

> 第二部・裝備升級　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 10 - 像管理原始碼一樣管理 Context：壓縮與 Token 最佳化.md`

## 今天要學會

1. 用 `EventsCompactionConfig` 設定壓縮
2. 換掉預設的摘要模型
3. **量出**壓縮前後的差異（原文只講設定，沒有數字）

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 問題：每一輪都在重送全部歷史

Day 09 講過，agent 的「記憶」就是把整包 `events` 重送給模型。
這代表 token 成本是**累積**的：

```
  第 1 輪：送 1 輪的內容
  第 2 輪：送 2 輪的內容
  第 10 輪：送 10 輪的內容   ← 已經很貴了
```

先把這件事量出來。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.apps import App
from google.adk.plugins.base_plugin import BasePlugin
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService


class TokenMeter(BasePlugin):
    """攔下每一次送出去的 request，記錄它有多少內容。"""

    def __init__(self, name: str = "token_meter"):
        super().__init__(name=name)
        self.rounds: list[dict] = []

    async def before_model_callback(self, *, callback_context, llm_request):
        contents = llm_request.contents or []
        chars = sum(
            len(p.text or "")
            for c in contents
            for p in (c.parts or [])
        )
        self.rounds.append({"contents": len(contents), "chars": chars})
        return None

    def report(self, label: str = "") -> None:
        print(f"{label}")
        for i, r in enumerate(self.rounds, 1):
            print(f"  第 {i:2d} 次模型呼叫: {r['contents']:3d} 段對話 / {r['chars']:6d} 字元")
        if self.rounds:
            print(f"  → 合計送出 {sum(r['chars'] for r in self.rounds):,} 字元")


TOPICS = [
    "用兩句話說明什麼是 REST API。",
    "那 GraphQL 跟它差在哪？",
    "gRPC 又是什麼？",
    "這三個我該選哪一個？",
    "如果團隊只有三個人呢？",
    "幫我整理成一個決策表。",
]

In [3]:
plain_meter = TokenMeter()
plain_agent = LlmAgent(
    name="plain", model=get_model(),
    instruction="你是技術顧問，用繁體中文回答，每次最多三句話。",
)
plain_runner = Runner(
    app=App(name="day10", root_agent=plain_agent, plugins=[plain_meter]),
    session_service=InMemorySessionService(),
)

sid = await new_session(plain_runner)
for q in TOPICS:
    await ask(plain_runner, q, session_id=sid)

plain_meter.report("【沒有壓縮】")

【沒有壓縮】
  第  1 次模型呼叫:   1 段對話 /     19 字元
  第  2 次模型呼叫:   3 段對話 /    149 字元
  第  3 次模型呼叫:   5 段對話 /    277 字元
  第  4 次模型呼叫:   7 段對話 /    396 字元
  第  5 次模型呼叫:   9 段對話 /    510 字元
  第  6 次模型呼叫:  11 段對話 /    610 字元
  → 合計送出 1,961 字元


每一輪送出的內容單調遞增。這就是長對話變貴的機制。

## 2. `EventsCompactionConfig`：把舊對話摘要掉

概念是：舊的事件不需要逐字保留，摘要成一段就好。

**⚠️ 設定掛在 `App` 上，不是 agent 上。** 掛錯地方完全沒有作用。

In [4]:
from google.adk.apps.app import EventsCompactionConfig

print("EventsCompactionConfig 的欄位：")
for f, info in EventsCompactionConfig.model_fields.items():
    print(f"  {f:22s} 預設 {info.default!r}")

EventsCompactionConfig 的欄位：
  summarizer             預設 None
  compaction_interval    預設 None
  overlap_size           預設 None
  token_threshold        預設 None
  event_retention_size   預設 None


| 欄位 | 意思 |
|---|---|
| `compaction_interval` | 每 N 個事件壓縮一次（**sliding window** 策略） |
| `token_threshold` | 超過 N tokens 才壓縮（**token-based** 策略） |
| `overlap_size` | 壓縮時保留多少重疊，避免語意斷裂 |
| `event_retention_size` | 最近幾個事件不壓縮（保持近期精確度） |
| `summarizer` | 用什麼來做摘要，不給就用預設 |

In [5]:
compact_meter = TokenMeter()
compact_agent = LlmAgent(
    name="compact", model=get_model(),
    instruction="你是技術顧問，用繁體中文回答，每次最多三句話。",
)
compact_app = App(
    name="day10c",
    root_agent=compact_agent,
    plugins=[compact_meter],
    events_compaction_config=EventsCompactionConfig(
        # sliding-window 這一對必須成雙出現，少一個會 ValidationError（第 4 節）
        compaction_interval=4,     # 每 4 個事件壓一次
        overlap_size=1,            # 保留 1 個重疊，避免語意在邊界斷掉
    ),
)
compact_runner = Runner(app=compact_app, session_service=InMemorySessionService())

sid_c = await new_session(compact_runner)
for q in TOPICS:
    await ask(compact_runner, q, session_id=sid_c)

compact_meter.report("【有壓縮】")

【有壓縮】
  第  1 次模型呼叫:   1 段對話 /     19 字元
  第  2 次模型呼叫:   3 段對話 /    144 字元
  第  3 次模型呼叫:   5 段對話 /    306 字元
  第  4 次模型呼叫:   7 段對話 /    428 字元
  第  5 次模型呼叫:   2 段對話 /   1276 字元
  第  6 次模型呼叫:   4 段對話 /   1414 字元
  → 合計送出 3,587 字元


## 3. 📌 把差異算出來

原文只給設定，沒有數字。數字才有說服力：

In [6]:
plain_total = sum(r["chars"] for r in plain_meter.rounds)
compact_total = sum(r["chars"] for r in compact_meter.rounds)

print(f"沒有壓縮 : {plain_total:,} 字元")
print(f"有壓縮   : {compact_total:,} 字元")
if plain_total:
    delta = (compact_total - plain_total) / plain_total * 100
    print(f"差異     : {delta:+.1f}%")

print(f"""
最後一次呼叫的大小：
  沒有壓縮 : {plain_meter.rounds[-1]['chars']:,} 字元 / {plain_meter.rounds[-1]['contents']} 段
  有壓縮   : {compact_meter.rounds[-1]['chars']:,} 字元 / {compact_meter.rounds[-1]['contents']} 段
""")

沒有壓縮 : 1,961 字元
有壓縮   : 3,587 字元
差異     : +82.9%

最後一次呼叫的大小：
  沒有壓縮 : 610 字元 / 11 段
  有壓縮   : 1,414 字元 / 4 段



### 怎麼看這個數字

對話越長，壓縮的效益越明顯。六輪對話可能還看不出差距（甚至因為多了摘要
的呼叫而略高），但三十輪、一百輪就是完全不同的量級。

**壓縮不是免費的**：它自己也要呼叫模型做摘要。所以：

- 短對話（< 10 輪）：不用開，開了反而虧
- 長對話 / 客服 / ambient agent：一定要開

## 4. ⚠️ 五個欄位其實是「兩對」

這是原文沒講、但一寫就會撞到的規則。那五個欄位**不能任意組合**：

| 這一對 | 策略 |
|---|---|
| `compaction_interval` + `overlap_size` | **sliding window**（固定間隔） |
| `token_threshold` + `event_retention_size` | **token-based**（超過門檻） |

規則有三條：

1. **每一對都是全有全無**——只給一個會被擋下來
2. **至少要給一對**，兩個都不給也會被擋
3. 兩對可以同時給

直接把所有組合跑一遍：

In [7]:
COMBOS = [
    ("只給 compaction_interval", dict(compaction_interval=4)),
    ("sliding-window 一對", dict(compaction_interval=4, overlap_size=1)),
    ("只給 token_threshold", dict(token_threshold=8000)),
    ("token-based 一對", dict(token_threshold=8000, event_retention_size=2)),
    ("只給 event_retention_size", dict(event_retention_size=2)),
    ("兩對都給", dict(compaction_interval=4, overlap_size=1,
                     token_threshold=8000, event_retention_size=2)),
    ("什麼都不給", dict()),
]

for label, kw in COMBOS:
    try:
        EventsCompactionConfig(**kw)
        print(f"  ✅ {label}")
    except Exception as exc:
        reason = str(exc).split("Value error, ")[-1].split(" [type")[0]
        print(f"  ❌ {label}")
        print(f"     → {reason}")

  ❌ 只給 compaction_interval
     → compaction_interval and overlap_size must be set together.
  ✅ sliding-window 一對
  ❌ 只給 token_threshold
     → token_threshold and event_retention_size must be set together.
  ✅ token-based 一對
  ❌ 只給 event_retention_size
     → token_threshold and event_retention_size must be set together.
  ✅ 兩對都給
  ❌ 什麼都不給
     → At least one compaction trigger must be configured: the token-threshold pair or the sliding-window pair.


### 錯誤訊息很好懂，但只有跑過才知道

三種錯誤訊息分別是：

- `compaction_interval and overlap_size must be set together.`
- `token_threshold and event_retention_size must be set together.`
- `At least one compaction trigger must be configured: the token-threshold
  pair or the sliding-window pair.`

**好消息是它會擋下來**——不像 Day 12 那些安靜失效的設定，
這個寫錯會立刻拋 `ValidationError`。

### 怎麼選

| 你的情境 | 用哪一對 |
|---|---|
| 對話長度可預期（客服流程、固定步驟） | `compaction_interval` + `overlap_size` |
| 單則訊息長度差異大（會貼程式碼、貼文件） | `token_threshold` + `event_retention_size` |
| 想同時要「固定節奏」和「成本上限」 | 兩對都給 |

兩個「第二欄位」的意義也不同：

- `overlap_size`：壓縮時保留幾個重疊事件，**避免語意在邊界斷掉**
- `event_retention_size`：最近幾個事件**完全不壓縮**，保持近期精確度

## 5. 換掉摘要模型

摘要本身也是一次模型呼叫。用便宜的模型做摘要、貴的模型做正事，
是很划算的組合。

In [8]:
from google.adk.apps.llm_event_summarizer import LlmEventSummarizer
from google.adk.models.google_llm import Gemini

from shared import DEFAULT_MODEL

cheap_summarizer = LlmEventSummarizer(
    llm=Gemini(model=DEFAULT_MODEL),   # 摘要用便宜的
)

tuned = App(
    name="day10t",
    root_agent=LlmAgent(
        name="tuned",
        model=get_model(),              # 正事用主力模型
        instruction="你是技術顧問，用繁體中文回答。",
    ),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=4,
        overlap_size=1,
        summarizer=cheap_summarizer,
    ),
)
print("摘要模型 :", tuned.events_compaction_config.summarizer.__class__.__name__)
print("主力模型 :", tuned.root_agent.model.model)

摘要模型 : LlmEventSummarizer
主力模型 : gemini-flash-lite-latest


### 自訂摘要 prompt

`LlmEventSummarizer` 接受 `prompt_template`。這在特定領域很有用——
例如客服系統只想保留「客戶抱怨什麼、承諾了什麼」。

In [9]:
import inspect

print("LlmEventSummarizer 簽章:", inspect.signature(LlmEventSummarizer.__init__))

domain_summarizer = LlmEventSummarizer(
    llm=Gemini(model=DEFAULT_MODEL),
    prompt_template=(
        "把以下客服對話濃縮成三行，只保留：\n"
        "1. 客戶的問題\n"
        "2. 我方已承諾的事項\n"
        "3. 尚未解決的部分\n"
        "其他寒暄一律略過。\n\n"
        "{events}"
    ),
)
print("\n✅ 自訂 prompt 的 summarizer 建立成功")

LlmEventSummarizer 簽章: (self, llm: 'BaseLlm', prompt_template: 'Optional[str]' = None)

✅ 自訂 prompt 的 summarizer 建立成功


## 6. ⚠️ 設在 agent 上是沒有用的

這一點值得實測，因為錯了不會報錯——只是安靜地沒有效果。

In [10]:
print("App 可設定的欄位:")
for f in App.model_fields:
    print(f"  {f}")

print("\nLlmAgent 有 events_compaction_config 嗎？",
      "events_compaction_config" in LlmAgent.model_fields)

App 可設定的欄位:
  name
  root_agent
  plugins
  events_compaction_config
  context_cache_config
  resumability_config

LlmAgent 有 events_compaction_config 嗎？ False


`LlmAgent` 根本沒有這個欄位。Pydantic 預設會擋掉未知欄位：

In [11]:
try:
    LlmAgent(
        name="wrong", model=get_model(), instruction="i",
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=4, overlap_size=1
        ),
    )
    print("⚠️ 沒有報錯——那就更要小心，設定會被安靜忽略")
except Exception as exc:
    print(f"✅ 被擋下來了：{type(exc).__name__}")
    print(f"   {str(exc)[:180]}")

✅ 被擋下來了：ValidationError
   1 validation error for LlmAgent
events_compaction_config
  Extra inputs are not permitted [type=extra_forbidden, input_value=EventsCompactionConfig(su...ent_retention_size=None), i


## 7. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 設了壓縮但沒效果 | 設在 agent 上了，要設在 **`App`** 上 |
| `ValidationError: ... must be set together` | 五個欄位其實是**兩對**，每對全有全無 |
| `At least one compaction trigger must be configured` | 兩對一個都沒給 |
| 開了壓縮反而更貴 | 對話太短，摘要的呼叫成本蓋過節省 |
| 壓縮後 agent 忘記重要細節 | `event_retention_size` 太小，或摘要 prompt 沒保住關鍵欄位 |
| 語意在壓縮邊界斷掉 | `overlap_size` 設 0，加一點重疊 |

## 8. 動手練習

1. 把 `TOPICS` 加到 20 題重跑，看壓縮的效益怎麼隨長度變化。
2. 改用 token-based 那一對（`token_threshold=2000, event_retention_size=2`），
   比較它跟 sliding window 的壓縮時機有什麼不同。
3. 用 `domain_summarizer` 跑一段客服對話，把摘要印出來，
   確認它真的只保留了那三項。

## 本日回顧

- **長對話變貴的根源**：每一輪都重送全部 `events`（Day 09）。
- **⚠️ 壓縮設定掛在 `App` 上**，`LlmAgent` 根本沒有這個欄位。
- **⚠️ 五個欄位其實是兩對**：`compaction_interval`+`overlap_size`（sliding window）、
  `token_threshold`+`event_retention_size`（token-based）。
  每對全有全無，至少要給一對——寫錯會直接 `ValidationError`。
- **壓縮不是免費的**——它自己要呼叫模型。短對話不划算。
- **摘要可以用便宜的模型**（`LlmEventSummarizer(llm=...)`），
  也可以用 `prompt_template` 針對領域客製。

---
**下一天 → `../day11_caching_and_artifacts/`**